In [2]:
import torch
import torch.nn as nn
import time
import numpy as np

# 1. Architecture MLP équivalente à NeuroDSL
class MnistNetPy(nn.Module):
    def __init__(self, d_in=784, d_hidden=128, d_out=10):
        super().__init__()
        # Utilisation de bias=False pour correspondre strictement à votre implémentation
        self.w1 = nn.Linear(d_in, d_hidden, bias=False)
        self.w2 = nn.Linear(d_hidden, d_out, bias=False)
        
    def forward(self, x):
        h = torch.relu(self.w1(x))
        return self.w2(h)

# 2. Setup du benchmark
device = torch.device('cuda')
model = MnistNetPy().to(device)
# AdamW avec les mêmes paramètres que dans votre code Julia
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, betas=(0.9, 0.999))
loss_fn = nn.CrossEntropyLoss()

# Simulation de données (100 batchs de 128 images)
B = 128
x_train = [torch.randn(B, 784).to(device) for _ in range(100)]
y_train = [torch.randint(0, 10, (B,)).to(device) for _ in range(100)]

# 3. Boucle de mesure (Jitter et Débit)
times = []

# Warmup pour stabiliser le cache CUDA
for i in range(10):
    optimizer.zero_grad()
    loss = loss_fn(model(x_train[i]), y_train[i])
    loss.backward()
    optimizer.step()

# Benchmark avec synchronisation stricte pour mesurer le "Jitter"
for i in range(100):
    torch.cuda.synchronize()
    start = time.perf_counter()
    
    optimizer.zero_grad()
    loss = loss_fn(model(x_train[i]), y_train[i])
    loss.backward()
    optimizer.step()
    
    torch.cuda.synchronize()
    times.append((time.perf_counter() - start) * 1000) # ms

print(f"--- PyTorch (MLP 784-128-10) ---")
print(f"Moyenne : {np.mean(times):.4f} ms")
print(f"Écart-type (Jitter) : {np.std(times):.4f} ms")

--- PyTorch (MLP 784-128-10) ---
Moyenne : 2.3266 ms
Écart-type (Jitter) : 0.3564 ms


In [5]:
import torch
import time
import numpy as np

# Définition d'un modèle simple de N couches
class SimpleNet(torch.nn.Module):
    def __init__(self, layers):
        super().__init__()
        # On garde une entrée à 784, une dimension cachée à 128, et sortie à 10
        module_list = [torch.nn.Linear(784, 128), torch.nn.ReLU()]
        
        # On ajoute les couches intermédiaires (toutes de 128 à 128)
        for _ in range(layers - 1):
            module_list.append(torch.nn.Linear(128, 128))
            module_list.append(torch.nn.ReLU())
            
        # Couche de sortie
        module_list.append(torch.nn.Linear(128, 10))
        self.net = torch.nn.Sequential(*module_list)

    def forward(self, x):
        return self.net(x)

def benchmark_pytorch(layers, iterations=100):
    device = torch.device('cuda')
    model = SimpleNet(layers).to(device).eval()
    input_data = torch.randn(128, 784).to(device)
    
    # Warm-up (essentiel pour PyTorch JIT)
    for _ in range(100):
        _ = model(input_data)
        torch.cuda.synchronize()
        
    mesures = []
    for _ in range(iterations):
        start = time.perf_counter()
        _ = model(input_data)
        torch.cuda.synchronize()
        mesures.append((time.perf_counter() - start) * 1000)
        
    # Filtrage similaire au vôtre (95ème percentile)
    mesures = np.array(mesures)
    mesures_filtrees = mesures[mesures < np.percentile(mesures, 95)]
    return np.mean(mesures_filtrees), np.std(mesures_filtrees)

# Lancement pour les mêmes paliers
for nb in [1, 2, 4, 8, 16]:
    m, s = benchmark_pytorch(nb)
    print(f"Couches: {nb} | Latence: {m:.4f} ms | σ: {s:.4f}")

Couches: 1 | Latence: 0.2531 ms | σ: 0.0093
Couches: 2 | Latence: 0.4149 ms | σ: 0.0099
Couches: 4 | Latence: 0.6785 ms | σ: 0.0320
Couches: 8 | Latence: 1.2097 ms | σ: 0.0576
Couches: 16 | Latence: 2.3491 ms | σ: 0.0790


In [2]:
"""
Benchmark mémoire GPU — PyTorch
Miroir exact du notebook NeuroDSL (cellules 21, 46–51)

Architectures testées :
  1. Linear(D,D) → LayerNorm → Linear(D,D)  [même que cellule 21]
  2. Transformer block complet               [même que bench_transformer_block]
  3. Gradient checkpointing                  [même que cellules 46–51]

Lance dans une cellule %%python de ton notebook Julia, ou dans un
kernel Python séparé.
"""

import torch
import torch.nn as nn
import torch.utils.checkpoint as ckpt
import gc
import math
import time
# ── Setup ────────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA requis"
DEVICE = torch.device("cuda")
print(f"PyTorch {torch.__version__}")
print(f"GPU     {torch.cuda.get_device_name(0)}")
print(f"VRAM    {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB\n")

# ── Helpers ──────────────────────────────────────────────────────────────────
def alloc_mb():
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / 1024**2

def peak_mb():
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1024**2

def reset():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def warmup_pass(fn):
    """Un pass complet pour déclencher la compilation JIT CUDA avant mesure."""
    with torch.no_grad():
        fn()
    reset()

# ── Architectures ─────────────────────────────────────────────────────────────

class LinearModel(nn.Module):
    """Linear(D,D) → LayerNorm → Linear(D,D) — miroir du modèle NeuroDSL cellule 21."""
    def __init__(self, D):
        super().__init__()
        self.fc1  = nn.Linear(D, D, bias=False)
        self.norm = nn.LayerNorm(D)
        self.fc2  = nn.Linear(D, D, bias=False)
    def forward(self, x):
        return self.fc2(self.norm(self.fc1(x)))


def rmsnorm(x, gamma, eps=1e-6):
    """RMSNorm manuel — évite la dépendance à nn.RMSNorm (PyTorch >= 2.4 seulement)."""
    rms_inv = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    return x * rms_inv * gamma


class TransformerBlock(nn.Module):
    """
    Même architecture que build_transformer_block_neuro :
      RMSNorm → QKV → softmax attention → résiduel
      RMSNorm → SwiGLU MLP → résiduel
    Input : (seq, dim) — tenseur 2D, identique à NeuroDSL.
    """
    def __init__(self, dim):
        super().__init__()
        self.gamma1 = nn.Parameter(torch.ones(dim))
        self.gamma2 = nn.Parameter(torch.ones(dim))
        # Poids comme matrices plates (dim×dim), trans_b implicite via .T
        for name in ["Wq", "Wk", "Wv", "Wo", "W1", "W2", "W3"]:
            setattr(self, name, nn.Parameter(torch.randn(dim, dim) * 0.01))

    def forward(self, x):
        # ── Attention ──────────────────────────────────────────────────────
        xn1   = rmsnorm(x, self.gamma1)
        q     = xn1 @ self.Wq.T
        k     = xn1 @ self.Wk.T
        v     = xn1 @ self.Wv.T
        scale = 1.0 / math.sqrt(q.size(-1))
        att   = torch.softmax(q @ k.T * scale, dim=-1)
        x     = x + att @ v @ self.Wo.T
        # ── MLP SwiGLU ─────────────────────────────────────────────────────
        xn2   = rmsnorm(x, self.gamma2)
        gate  = xn2 @ self.W1.T
        up    = xn2 @ self.W3.T
        swiglu = gate * torch.sigmoid(gate) * up
        x     = x + swiglu @ self.W2.T
        return x


class DeepNet(nn.Module):
    """Réseau profond pour le bench checkpointing — miroir cellules 46–51."""
    def __init__(self, depth, dim):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(depth)])

    def forward_full(self, x):
        """Conserve toutes les activations (baseline)."""
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

    def forward_ckpt(self, x, every=3):
        """Checkpointing : recompute 1 couche sur `every`."""
        for i, layer in enumerate(self.layers):
            if i % every == 0:
                x = ckpt.checkpoint(layer, x, use_reentrant=False)
            else:
                x = torch.relu(layer(x))
        return x


# ═══════════════════════════════════════════════════════════════════════════════
# BENCHMARK 1 — Linear model, même configs que NeuroDSL cellule 21
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 62)
print("   CUDA — delta mémoire GPU — Linear→LayerNorm→Linear")
print("=" * 62)
print(f"{'config (M×D)':<20} {'avant (MB)':>10} {'après (MB)':>10} {'delta (MB)':>10}")
print("-" * 62)

configs_linear = [(8, 64), (32, 128), (128, 256), (512, 512), (1024, 512)]
results_linear_pt  = {}

for M, D in configs_linear:
    reset()
    model = LinearModel(D).to(DEVICE)
    x = torch.randn(M, D, device=DEVICE)
    y = torch.zeros(M, D, device=DEVICE)

    # Warmup : déclenche la compilation CUDA JIT avant mesure
    warmup_pass(lambda: nn.functional.mse_loss(model(x), y))


    model.zero_grad()
    torch.cuda.synchronize()
    avant = alloc_mb()

    out  = model(x)
    loss = nn.functional.mse_loss(out, y)
    loss.backward()
    torch.cuda.synchronize()
    apres = alloc_mb()
    delta = apres - avant

    results_linear_pt[(M, D)] = delta
    print(f"  {M}×{D:<17} {avant:>10.2f} {apres:>10.2f} {delta:>10.2f}")

    del model, x, y, out, loss
    reset()


# ═══════════════════════════════════════════════════════════════════════════════
# BENCHMARK 2 — Transformer block, dim 256 / 512 / 1024
# ═══════════════════════════════════════════════════════════════════════════════
print()
print("=" * 62)
print("   CUDA — delta mémoire GPU — Transformer block (seq=64)")
print("=" * 62)
print(f"{'config':<20} {'avant (MB)':>10} {'après (MB)':>10} {'delta (MB)':>10} {'pic (MB)':>10}")
print("-" * 62)

results_tb_pt = {}

import time

for dim in [256, 512, 1024]:
    seq = 64
    reset()
    block = TransformerBlock(dim).to(DEVICE)
    x = torch.randn(seq, dim, device=DEVICE)
    y = torch.zeros(seq, dim, device=DEVICE)

    # Warmup (inchangé)
    warmup_pass(lambda: nn.functional.mse_loss(block(x), y))

    # --- Mesure de vitesse (médiane sur 50 itérations) ---
    times = []
    for _ in range(50):
        block.zero_grad()
        torch.cuda.synchronize()
        start = time.perf_counter()
        out = block(x)
        loss = nn.functional.mse_loss(out, y)
        loss.backward()
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) * 1000  # ms
        times.append(elapsed)

    median_time = sorted(times)[len(times)//2]

    # --- Mesure mémoire (inchangée) ---
    block.zero_grad()
    torch.cuda.synchronize()
    avant = alloc_mb()

    out  = block(x)
    loss = nn.functional.mse_loss(out, y)
    loss.backward()
    torch.cuda.synchronize()
    apres = alloc_mb()
    pic   = peak_mb()
    delta = apres - avant

    results_tb_pt[dim] = {"avant": avant, "apres": apres, "delta": delta, "pic": pic, "time_ms": median_time}
    print(f"  dim={dim:<16} temps médian : {median_time:.3f} ms  |  pic mémoire : {pic:.2f} MB")

    del block, x, y, out, loss
    reset()

# ═══════════════════════════════════════════════════════════════════════════════
# BENCHMARK 3 — Gradient checkpointing (miroir cellules 46–51)
# ═══════════════════════════════════════════════════════════════════════════════
print()
print("=" * 62)
print("   Gradient checkpointing — économie mémoire (PyTorch)")
print("=" * 62)

ckpt_configs = [
    (10, 1024, 3, 64),   # depth, dim, every, batch — miroir NeuroDSL depth=10
    (20, 2048, 4, 64),   # miroir NeuroDSL depth=20
]

results_ckpt_pt = []

for depth, dim, every, batch in ckpt_configs:
    print(f"\n  depth={depth}  dim={dim}  every={every}")
    reset()
    model = DeepNet(depth, dim).to(DEVICE)
    x = torch.randn(batch, dim, device=DEVICE)
    y = torch.zeros(batch, dim, device=DEVICE)

    # Sans checkpointing
    warmup_pass(lambda: nn.functional.mse_loss(model.forward_full(x), y))
    model.zero_grad(); reset()

    out  = model.forward_full(x)
    loss = nn.functional.mse_loss(out, y)
    loss.backward()
    pic_full = peak_mb()
    del out, loss
    reset()

    # Avec checkpointing
    model.zero_grad(); reset()
    out  = model.forward_ckpt(x, every=every)
    loss = nn.functional.mse_loss(out, y)
    loss.backward()
    pic_ckpt = peak_mb()

    eco = (pic_full - pic_ckpt) / pic_full * 100
    results_ckpt_pt.append({
        "depth": depth, "dim": dim, "every": every,
        "full_mb": pic_full, "ckpt_mb": pic_ckpt, "eco_pct": eco,
    })

    print(f"    Baseline     : {pic_full:.2f} MB")
    print(f"    Checkpointed : {pic_ckpt:.2f} MB")
    print(f"    Économie     : {pic_full - pic_ckpt:.2f} MB ({eco:.1f}%)")

    del model, x, y, out, loss
    reset()


# ═══════════════════════════════════════════════════════════════════════════════
# RÉCAPITULATIF — NeuroDSL vs PyTorch (delta MB, même configs)
# ═══════════════════════════════════════════════════════════════════════════════

# Valeurs NeuroDSL tirées du notebook (cellule 21)
neurodsl_linear = {
    (8,   64):  0.16,
    (32, 128):  0.76,
    (128, 256): 4.01,
    (512, 512): 24.03,
}

neurodsl_ckpt = [
    {"depth": 10, "dim": 1024, "eco_pct": 33.3},
    {"depth": 20, "dim": 2048, "eco_pct": 42.9},
]

print()
print("┌" + "─"*66 + "┐")
print("│{:^66}│".format("Comparaison delta mémoire GPU — NeuroDSL vs PyTorch"))
print("├" + "─"*12 + "┬" + "─"*14 + "┬" + "─"*14 + "┬" + "─"*14 + "┬" + "─"*9 + "┤")
print("│{:<12}│{:>14}│{:>14}│{:>14}│{:>9}│".format(
    "  config", "  NeuroDSL MB", "  PyTorch MB", "  diff MB", "  ratio"))
print("├" + "─"*12 + "┼" + "─"*14 + "┼" + "─"*14 + "┼" + "─"*14 + "┼" + "─"*9 + "┤")

for (M, D), nd in sorted(neurodsl_linear.items()):
    pt  = results_linear_pt.get((M, D), float("nan"))
    dif = pt - nd
    rat = pt / nd if nd != 0 else float("nan")
    tag = "= parity" if abs(rat-1) < 0.15 else ("lighter" if rat < 1 else "heavier")
    print("│{:<12}│{:>14.2f}│{:>14.2f}│{:>+14.2f}│{:>9}│".format(
        f"  {M}x{D}", nd, pt, dif, f"  {rat:.2f}x"))

print("└" + "─"*12 + "┴" + "─"*14 + "┴" + "─"*14 + "┴" + "─"*14 + "┴" + "─"*9 + "┘")

print()
print("┌" + "─"*60 + "┐")
print("│{:^60}│".format("Checkpointing — économie mémoire comparée"))
print("├" + "─"*20 + "┬" + "─"*18 + "┬" + "─"*18 + "┤")
print("│{:<20}│{:>18}│{:>18}│".format("  config", "  NeuroDSL éco", "  PyTorch éco"))
print("├" + "─"*20 + "┼" + "─"*18 + "┼" + "─"*18 + "┤")
for nd, pt in zip(neurodsl_ckpt, results_ckpt_pt):
    label = f"  d={nd['depth']} dim={nd['dim']}"
    print("│{:<20}│{:>18}│{:>18}│".format(
        label,
        f"  {nd['eco_pct']:.1f}%",
        f"  {pt['eco_pct']:.1f}%"))
print("└" + "─"*20 + "┴" + "─"*18 + "┴" + "─"*18 + "┘")

print("\nFin du benchmark PyTorch.")

PyTorch 2.5.1+cu121
GPU     NVIDIA RTX A5500 Laptop GPU
VRAM    16384 MB

   CUDA — delta mémoire GPU — Linear→LayerNorm→Linear
config (M×D)         avant (MB) après (MB) delta (MB)
--------------------------------------------------------------
  8×64                     16.29      16.32       0.04
  32×128                    16.41      16.56       0.16
  128×256                    17.00      17.75       0.75
  512×512                    20.25      24.26       4.00
  1024×512                    22.25      28.26       6.00

   CUDA — delta mémoire GPU — Transformer block (seq=64)
config               avant (MB) après (MB) delta (MB)   pic (MB)
--------------------------------------------------------------
  dim=256              temps médian : 5.095 ms  |  pic mémoire : 20.32 MB
  dim=512              temps médian : 5.138 ms  |  pic mémoire : 31.38 MB
  dim=1024             temps médian : 5.771 ms  |  pic mémoire : 74.51 MB

   Gradient checkpointing — économie mémoire (PyTorch)

  depth